In [ ]:
!pip -q install catboost openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

print("Setup complete")

Setup complete


In [ ]:
# ============================================================
# REAL BLACKBOX OBSERVATIONS — R1/R2
# No external dataset
# ============================================================

import pandas as pd

columns = [
    "age",
    "baseline_score",
    "comorbidity_ratio",
    "dependants",
    "prior_visits",
    "recent_admissions",
    "requested_beds",
    "vitals_index",
    "ward",
    "years_registered",
    "score"
]

rows = [
    # ---------------- PRIOR VISITS ----------------
    (73,773.5,0,5,0,0,100,100,"B",10.5,0.8787),
    (73,773.5,0,5,5,0,100,100,"B",10.5,0.9304),
    (73,773.5,0,5,10,0,100,100,"B",10.5,0.9567),
    (73,773.5,0,5,15,0,100,100,"B",10.5,0.9713),
    (73,773.5,0,5,20,0,100,100,"B",10.5,0.9793),

    # ---------------- YEARS REGISTERED ----------------
    (73,750,0,0,20,0,100,100,"B",0,0.9684),
    (73,750,0,0,20,0,100,100,"B",5,0.9768),
    (73,750,0,0,20,0,100,100,"B",7,0.9784),
    (73,750,0,0,20,0,100,100,"B",8,0.9784),
    (73,750,0,0,20,0,100,100,"B",10,0.9786),
    (73,750,0,0,20,0,100,100,"B",11,0.9778),
    (73,750,0,0,20,0,100,100,"B",12,0.9776),
    (73,750,0,0,20,0,100,100,"B",15,0.9765),
    (73,750,0,0,20,0,100,100,"B",20,0.9745),
    (73,750,0,0,20,0,100,100,"B",25,0.9708),
    (73,750,0,0,20,0,100,100,"B",30,0.9629),
    (73,750,0,0,20,0,100,100,"B",40,0.9308),

    # ---------------- BASELINE SCORE ----------------
    (73,700,0,0,20,0,100,100,"B",20,0.9714),
    (73,750,0,0,20,0,100,100,"B",20,0.9745),
    (73,775,0,0,20,0,100,100,"B",20,0.9744),
    (73,800,0,0,20,0,100,100,"B",20,0.9738),
    (73,850,0,0,20,0,100,100,"B",20,0.9717),
    (73,875,0,0,20,0,100,100,"B",20,0.9716),
    (73,900,0,0,20,0,100,100,"B",20,0.9703),

    # Fine baseline region
    (73,740,0,0,20,0,100,100,"B",10,0.9783),
    (73,750,0,0,20,0,100,100,"B",10,0.9786),
    (73,760,0,0,20,0,100,100,"B",10,0.9787),
    (73,765,0,0,20,0,100,100,"B",10,0.9786),
    (73,768,0,0,20,0,100,100,"B",10,0.9789),
    (73,770,0,0,20,0,100,100,"B",10,0.9788),
    (73,771,0,0,20,0,100,100,"B",10,0.9790),
    (73,772,0,0,20,0,100,100,"B",10,0.9790),
    (73,773,0,0,20,0,100,100,"B",10,0.9790),
    (73,774,0,0,20,0,100,100,"B",10,0.9790),
    (73,775,0,0,20,0,100,100,"B",10,0.9789),
    (73,780,0,0,20,0,100,100,"B",10,0.9785),

    # ---------------- AGE ----------------
    (71,750,0,0,20,0,100,100,"B",10,0.9771),
    (72,750,0,0,20,0,100,100,"B",10,0.9782),
    (73,750,0,0,20,0,100,100,"B",10,0.9786),
    (74,750,0,0,20,0,100,100,"B",10,0.9770),
    (75,750,0,0,20,0,100,100,"B",10,0.9769),

    # ---------------- COMORBIDITY ----------------
    (73,750,0.0,0,20,0,100,100,"B",10,0.9786),
    (73,750,0.1,0,20,0,100,100,"B",10,0.9758),
    (73,750,0.2,0,20,0,100,100,"B",10,0.9692),
    (73,750,0.3,0,20,0,100,100,"B",10,0.9565),
    (73,750,0.4,0,20,0,100,100,"B",10,0.9476),
    (73,750,1.0,0,20,0,100,100,"B",10,0.8177),

    # ---------------- RECENT ADMISSIONS ----------------
    (73,750,0,0,20,0,100,100,"B",10,0.9786),
    (73,750,0,0,20,1,100,100,"B",10,0.9782),
    (73,750,0,0,20,2,100,100,"B",10,0.9659),
    (73,750,0,0,20,3,100,100,"B",10,0.9595),
    (73,750,0,0,20,4,100,100,"B",10,0.9555),
    (73,750,0,0,20,5,100,100,"B",10,0.9406),

    # ---------------- WARD ----------------
    (73,750,0,0,20,0,100,100,"A",10,0.9763),
    (73,750,0,0,20,0,100,100,"B",10,0.9786),
    (73,750,0,0,20,0,100,100,"C",10,0.9750),
    (73,750,0,0,20,0,100,100,"D",10,0.9697),

    # ---------------- YEARS + WARD INTERACTION ----------------
    (73,750,0,0,20,0,100,100,"C",20,0.9736),
    (73,750,0,0,20,0,100,100,"D",20,0.9684),

    # ---------------- BEDS ----------------
    (73,750,0,0,20,0,0,100,"B",10,0.9786),
    (73,750,0,0,20,0,50,100,"B",10,0.9786),
    (73,750,0,0,20,0,100,100,"B",10,0.9786),

    # ---------------- VITALS ----------------
    (73,750,0,0,20,0,100,0,"B",10,0.9786),
    (73,750,0,0,20,0,100,50,"B",10,0.9786),
    (73,750,0,0,20,0,100,100,"B",10,0.9786),

    # ---------------- DEPENDANTS ----------------
    (73,772.725,0,0,20,0,100,100,"B",10.5,0.9793),
    (73,772.725,0,1,20,0,100,100,"B",10.5,0.9793),
    (73,772.725,0,2,20,0,100,100,"B",10.5,0.9793),
    (73,772.725,0,3,20,0,100,100,"B",10.5,0.9793),
    (73,772.725,0,4,20,0,100,100,"B",10.5,0.9793),
    (73,772.725,0,5,20,0,100,100,"B",10.5,0.9793),
    (73,772.725,0,6,20,0,100,100,"B",10.5,0.9793),

    # ---------------- COMBINED CHAMPION ----------------
    (73,773.5,0,5,20,0,100,100,"B",10.5,0.9793),
]

blackbox_data = pd.DataFrame(rows, columns=columns)

print("Dataset shape:", blackbox_data.shape)
display(blackbox_data.head(10))

Dataset shape: (73, 11)


,age,baseline_score,comorbidity_ratio,dependants,prior_visits,recent_admissions,requested_beds,vitals_index,ward,years_registered,score
0,73,773.5,0.0,5,0,0,100,100,B,10.5,0.8787
1,73,773.5,0.0,5,5,0,100,100,B,10.5,0.9304
2,73,773.5,0.0,5,10,0,100,100,B,10.5,0.9567
3,73,773.5,0.0,5,15,0,100,100,B,10.5,0.9713
4,73,773.5,0.0,5,20,0,100,100,B,10.5,0.9793
5,73,750.0,0.0,0,20,0,100,100,B,0.0,0.9684
6,73,750.0,0.0,0,20,0,100,100,B,5.0,0.9768
7,73,750.0,0.0,0,20,0,100,100,B,7.0,0.9784
8,73,750.0,0.0,0,20,0,100,100,B,8.0,0.9784
9,73,750.0,0.0,0,20,0,100,100,B,10.0,0.9786


In [ ]:
print("Missing values:")
print(blackbox_data.isna().sum())

print("\nScore range:")
print(blackbox_data["score"].min(), "to", blackbox_data["score"].max())

print("\nNumber of observations:", len(blackbox_data))

print("\nWard distribution:")
print(blackbox_data["ward"].value_counts())

Missing values:
age                  0
baseline_score       0
comorbidity_ratio    0
dependants           0
prior_visits         0
recent_admissions    0
requested_beds       0
vitals_index         0
ward                 0
years_registered     0
score                0
dtype: int64

Score range:
0.8177 to 0.9793

Number of observations: 73

Ward distribution:
ward
B    68
C     2
D     2
A     1
Name: count, dtype: int64


In [ ]:
blackbox_data.to_csv("real_blackbox_r1_r2.csv", index=False)

print("Saved real BLACKBOX data:", len(blackbox_data), "rows")

Saved real BLACKBOX data: 73 rows


In [ ]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()

Saving blackbox_R1_R2_R4_combined.csv to blackbox_R1_R2_R4_combined.csv


In [ ]:
df = pd.read_csv("blackbox_R1_R2_R4_combined.csv")

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nDecision distribution:")
print(df["decision"].value_counts())

print("\nScore distribution:")
print(df["score"].describe())

display(df.head())

Shape: (129, 13)

Columns:
['round_id', 'score', 'decision', 'age', 'baseline_score', 'comorbidity_ratio', 'dependants', 'prior_visits', 'recent_admissions', 'requested_beds', 'vitals_index', 'ward', 'years_registered']

Decision distribution:
decision
APPROVE    109
DECLINE     20
Name: count, dtype: int64

Score distribution:
count    129.000000
mean       0.768898
std        0.259378
min        0.053000
25%        0.643000
50%        0.878700
75%        0.978800
max        0.979300
Name: score, dtype: float64


,round_id,score,decision,age,baseline_score,comorbidity_ratio,dependants,prior_visits,recent_admissions,requested_beds,vitals_index,ward,years_registered
0,R4 · 80,0.0530,DECLINE,60.216,356.147,0.480,5.934,8.328,2.168,33.374,84.756,D,7.347
1,R4 · 79,0.3942,DECLINE,54.653,438.932,0.994,4.537,15.191,2.469,80.548,29.755,D,34.101
2,R4 · 78,0.6343,APPROVE,36.010,845.492,0.748,2.851,18.406,1.652,80.202,31.836,C,38.272
3,R4 · 77,0.3479,DECLINE,60.408,467.448,0.791,2.243,1.296,4.324,5.812,35.493,D,23.094
4,R4 · 75,0.7902,APPROVE,63.529,823.773,0.247,0.078,3.503,3.587,5.249,91.464,A,10.470


In [ ]:
# ============================================================
# BLACKBOX RECONSTRUCTION - MODEL COMPARISON
# ============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Features
# ------------------------------------------------------------

feature_cols = [
    "age",
    "baseline_score",
    "comorbidity_ratio",
    "dependants",
    "prior_visits",
    "recent_admissions",
    "requested_beds",
    "vitals_index",
    "ward",
    "years_registered"
]

X = df[feature_cols].copy()
y = df["score"].copy()

# Make ward categorical for CatBoost
X_cat = X.copy()
X_cat["ward"] = X_cat["ward"].astype(str)

# ------------------------------------------------------------
# 2. 5-Fold stratified split
#    Stratify using APPROVE/DECLINE so every fold contains
#    some lower-score observations.
# ------------------------------------------------------------

strata = df["decision"]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ------------------------------------------------------------
# 3. Models
# ------------------------------------------------------------

models = {
    "CatBoost": CatBoostRegressor(
        iterations=500,
        depth=5,
        learning_rate=0.03,
        loss_function="RMSE",
        l2_leaf_reg=5,
        random_seed=42,
        verbose=False
    ),

    "ExtraTrees": Pipeline([
        ("encoder", ColumnTransformer([
            ("ward", OneHotEncoder(handle_unknown="ignore"), ["ward"])
        ], remainder="passthrough")),

        ("model", ExtraTreesRegressor(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "RandomForest": Pipeline([
        ("encoder", ColumnTransformer([
            ("ward", OneHotEncoder(handle_unknown="ignore"), ["ward"])
        ], remainder="passthrough")),

        ("model", RandomForestRegressor(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

# ------------------------------------------------------------
# 4. Cross-validation
# ------------------------------------------------------------

results = []

for name, model in models.items():

    rmses = []
    maes = []
    r2s = []
    decision_accs = []

    for train_idx, test_idx in skf.split(X, strata):

        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        if name == "CatBoost":
            X_train["ward"] = X_train["ward"].astype(str)
            X_test["ward"] = X_test["ward"].astype(str)

            model.fit(
                X_train,
                y_train,
                cat_features=["ward"]
            )
        else:
            model.fit(X_train, y_train)

        pred = model.predict(X_test)

        # Keep score inside valid range
        pred = np.clip(pred, 0, 1)

        rmse = np.sqrt(mean_squared_error(y_test, pred))
        mae = mean_absolute_error(y_test, pred)
        r2 = r2_score(y_test, pred)

        # Decision threshold for now
        # We will optimize this later.
        predicted_decision = np.where(
            pred >= 0.5,
            "APPROVE",
            "DECLINE"
        )

        actual_decision = df.iloc[test_idx]["decision"].values

        acc = np.mean(predicted_decision == actual_decision)

        rmses.append(rmse)
        maes.append(mae)
        r2s.append(r2)
        decision_accs.append(acc)

    results.append({
        "Model": name,
        "RMSE": np.mean(rmses),
        "MAE": np.mean(maes),
        "R2": np.mean(r2s),
        "Decision Accuracy": np.mean(decision_accs)
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "RMSE",
    ascending=True
).reset_index(drop=True)

display(results_df)

,Model,RMSE,MAE,R2,Decision Accuracy
0,ExtraTrees,0.111083,0.064695,0.810001,0.953538
1,CatBoost,0.123456,0.068492,0.761203,0.892000
2,RandomForest,0.127550,0.068805,0.748580,0.922769


In [ ]:
# ============================================================
# FIND BEST DECISION THRESHOLD
# ============================================================

# Train CatBoost on all available data
final_cat = CatBoostRegressor(
    iterations=700,
    depth=5,
    learning_rate=0.03,
    loss_function="RMSE",
    l2_leaf_reg=5,
    random_seed=42,
    verbose=False
)

X_full = df[feature_cols].copy()
X_full["ward"] = X_full["ward"].astype(str)

final_cat.fit(
    X_full,
    y,
    cat_features=["ward"]
)

full_pred = np.clip(
    final_cat.predict(X_full),
    0,
    1
)

threshold_results = []

for threshold in np.arange(0.30, 0.71, 0.01):

    predicted = np.where(
        full_pred >= threshold,
        "APPROVE",
        "DECLINE"
    )

    accuracy = np.mean(
        predicted == df["decision"].values
    )

    threshold_results.append(
        [threshold, accuracy]
    )

threshold_df = pd.DataFrame(
    threshold_results,
    columns=["threshold", "accuracy"]
)

best_threshold = threshold_df.loc[
    threshold_df["accuracy"].idxmax()
]

print("Best threshold:")
print(best_threshold)

display(
    threshold_df.sort_values(
        "accuracy",
        ascending=False
    ).head(10)
)

Best threshold:
threshold    0.480000
accuracy     0.992248
Name: 18, dtype: float64


,threshold,accuracy
18,0.48,0.992248
21,0.51,0.984496
17,0.47,0.984496
19,0.49,0.984496
20,0.50,0.984496
23,0.53,0.984496
22,0.52,0.984496
16,0.46,0.984496
15,0.45,0.976744
13,0.43,0.968992


In [ ]:
# ============================================================
# FIX + EXTRA TREES ERROR ANALYSIS
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# Features
feature_cols = [
    "age",
    "baseline_score",
    "comorbidity_ratio",
    "dependants",
    "prior_visits",
    "recent_admissions",
    "requested_beds",
    "vitals_index",
    "ward",
    "years_registered"
]

X = df[feature_cols].copy()
y = df["score"].copy()

# ExtraTrees model
best_model = Pipeline([
    ("encoder", ColumnTransformer([
        ("ward", OneHotEncoder(handle_unknown="ignore"), ["ward"])
    ], remainder="passthrough")),

    ("model", ExtraTreesRegressor(
        n_estimators=1000,
        max_depth=8,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
best_model.fit(X, y)

# Predict
pred = np.clip(best_model.predict(X), 0, 1)

# Create evaluation dataframe
df_eval = df.copy()
df_eval["predicted_score"] = pred
df_eval["score_error"] = df_eval["score"] - df_eval["predicted_score"]
df_eval["abs_error"] = abs(df_eval["score_error"])

# Metrics
print("===== EXTRA TREES =====")
print("RMSE:", np.sqrt(mean_squared_error(y, pred)))
print("MAE :", mean_absolute_error(y, pred))
print("R2  :", r2_score(y, pred))
print("Max absolute error:", df_eval["abs_error"].max())

# Worst predictions
print("\n===== WORST 20 PREDICTIONS =====")

display(
    df_eval.sort_values(
        "abs_error",
        ascending=False
    )[[
        "round_id",
        "score",
        "predicted_score",
        "score_error",
        "abs_error",
        "age",
        "baseline_score",
        "comorbidity_ratio",
        "dependants",
        "prior_visits",
        "recent_admissions",
        "requested_beds",
        "vitals_index",
        "ward",
        "years_registered"
    ]].head(20)
)

===== EXTRA TREES =====
RMSE: 0.03066765391159518
MAE : 0.015405170702132473
R2  : 0.9859112324762845
Max absolute error: 0.18737762499999813

===== WORST 20 PREDICTIONS =====


,round_id,score,predicted_score,score_error,abs_error,age,baseline_score,comorbidity_ratio,dependants,prior_visits,recent_admissions,requested_beds,vitals_index,ward,years_registered
55,R4 · 13,0.0530,0.240378,-0.187378,0.187378,37.895,379.038,0.062,5.107,10.499,1.529,27.348,94.871,D,9.282
25,R4 · 49,0.8173,0.695258,0.122042,0.122042,58.262,473.524,0.090,3.076,9.724,4.614,75.408,97.515,D,11.306
8,R4 · 71,0.1411,0.251613,-0.110513,0.110513,33.143,342.608,0.728,4.647,0.531,3.543,73.419,94.937,A,25.643
36,R4 · 33,0.0530,0.137094,-0.084094,0.084094,18.480,385.381,0.904,2.888,14.684,2.399,61.910,30.078,D,15.516
1,R4 · 79,0.3942,0.321877,0.072323,0.072323,54.653,438.932,0.994,4.537,15.191,2.469,80.548,29.755,D,34.101
6,R4 · 73,0.0530,0.124568,-0.071568,0.071568,54.420,357.339,0.318,1.799,9.973,3.865,63.666,34.293,D,18.923
10,R4 · 68,0.5378,0.467810,0.069990,0.069990,63.766,425.686,0.640,0.342,6.536,4.482,47.570,3.882,D,12.560
58,R4 · 10,0.8650,0.803131,0.061869,0.061869,57.993,682.322,0.022,4.713,1.745,1.399,15.873,22.513,D,13.864
32,R4 · 37,0.6912,0.638622,0.052578,0.052578,18.209,870.374,0.297,1.240,15.602,3.878,44.838,79.473,D,22.631
13,R4 · 64,0.3175,0.365039,-0.047539,0.047539,18.159,878.286,0.926,3.482,11.396,3.279,92.010,62.040,C,24.708


In [ ]:
# ============================================================
# PROPER HOLDOUT VALIDATION
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score
)
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

feature_cols = [
    "age",
    "baseline_score",
    "comorbidity_ratio",
    "dependants",
    "prior_visits",
    "recent_admissions",
    "requested_beds",
    "vitals_index",
    "ward",
    "years_registered"
]

X = df[feature_cols]
y = df["score"]

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Model
preprocessor = ColumnTransformer(
    [
        ("ward", OneHotEncoder(handle_unknown="ignore"), ["ward"])
    ],
    remainder="passthrough"
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(
        n_estimators=1000,
        max_depth=8,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

# Train ONLY on training data
model.fit(X_train, y_train)

# Test prediction
test_pred = np.clip(model.predict(X_test), 0, 1)

print("===== HOLDOUT TEST RESULTS =====")
print("Train rows:", len(X_train))
print("Test rows :", len(X_test))

print("\nRMSE:", np.sqrt(mean_squared_error(y_test, test_pred)))
print("MAE :", mean_absolute_error(y_test, test_pred))
print("R2  :", r2_score(y_test, test_pred))

# ------------------------------------------------------------
# Decision threshold
# ------------------------------------------------------------

actual_decision = (y_test >= 0.50).astype(int)

thresholds = np.arange(0.40, 0.61, 0.01)

best_threshold = 0.50
best_accuracy = 0

for threshold in thresholds:
    predicted_decision = (test_pred >= threshold).astype(int)
    acc = accuracy_score(actual_decision, predicted_decision)

    if acc > best_accuracy:
        best_accuracy = acc
        best_threshold = threshold

print("\nBest threshold:", round(best_threshold, 2))
print("Decision accuracy:", best_accuracy)

# Detailed predictions
test_results = X_test.copy()
test_results["actual_score"] = y_test
test_results["predicted_score"] = test_pred
test_results["error"] = test_results["actual_score"] - test_results["predicted_score"]
test_results["abs_error"] = abs(test_results["error"])

display(
    test_results.sort_values(
        "abs_error",
        ascending=False
    )
)

===== HOLDOUT TEST RESULTS =====
Train rows: 103
Test rows : 26

RMSE: 0.15939867635783
MAE : 0.07982665485144665
R2  : 0.7134859062013432

Best threshold: 0.47
Decision accuracy: 0.8461538461538461


,age,baseline_score,comorbidity_ratio,dependants,prior_visits,recent_admissions,requested_beds,vitals_index,ward,years_registered,actual_score,predicted_score,error,abs_error
55,37.895,379.038,0.062,5.107,10.499,1.529,27.348,94.871,D,9.282,0.0530,0.720017,-0.667017,0.667017
36,18.480,385.381,0.904,2.888,14.684,2.399,61.910,30.078,D,15.516,0.0530,0.295569,-0.242569,0.242569
10,63.766,425.686,0.640,0.342,6.536,4.482,47.570,3.882,D,12.560,0.5378,0.329376,0.208424,0.208424
18,32.636,754.846,0.377,5.218,14.550,4.514,68.346,19.934,D,25.686,0.6464,0.450238,0.196162,0.196162
19,45.753,774.367,0.859,4.011,1.229,4.019,62.212,31.191,C,18.009,0.3202,0.466039,-0.145839,0.145839
4,63.529,823.773,0.247,0.078,3.503,3.587,5.249,91.464,A,10.470,0.7902,0.665769,0.124431,0.124431
11,50.904,752.336,0.879,5.378,2.611,4.744,38.282,35.386,A,8.573,0.3475,0.453819,-0.106319,0.106319
56,34.668,364.298,0.518,1.057,7.558,2.619,46.564,70.260,C,13.063,0.4852,0.587395,-0.102195,0.102195
27,36.166,722.916,0.898,1.725,18.711,0.872,71.638,4.811,C,19.967,0.6400,0.549044,0.090956,0.090956
64,32.667,744.397,0.659,5.861,17.086,0.355,90.761,81.098,C,28.480,0.7143,0.644423,0.069877,0.069877


In [ ]:
# ============================================================
# MODEL COMPARISON - IMPROVED FEATURES
# ============================================================

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# FEATURES
# ------------------------------------------------------------

features = [
    "age",
    "baseline_score",
    "comorbidity_ratio",
    "dependants",
    "prior_visits",
    "recent_admissions",
    "requested_beds",
    "vitals_index",
    "ward",
    "years_registered"
]

X = df[features].copy()
y = df["score"].copy()

# ------------------------------------------------------------
# FEATURE ENGINEERING
# ------------------------------------------------------------

def engineer_features(data):

    d = data.copy()

    # Nonlinear effects
    d["age_sq"] = d["age"] ** 2
    d["baseline_sq"] = d["baseline_score"] ** 2
    d["comorbidity_sq"] = d["comorbidity_ratio"] ** 2
    d["recent_admissions_sq"] = d["recent_admissions"] ** 2
    d["prior_visits_sq"] = d["prior_visits"] ** 2
    d["years_registered_sq"] = d["years_registered"] ** 2

    # Centered nonlinear terms
    d["age_centered_sq"] = (d["age"] - 73) ** 2
    d["baseline_centered_sq"] = (d["baseline_score"] - 773) ** 2
    d["years_centered_sq"] = (d["years_registered"] - 10.5) ** 2

    # Useful interactions
    d["comorbidity_recent"] = (
        d["comorbidity_ratio"] *
        d["recent_admissions"]
    )

    d["age_baseline"] = (
        d["age"] *
        d["baseline_score"]
    )

    d["prior_recent"] = (
        d["prior_visits"] *
        d["recent_admissions"]
    )

    d["baseline_years"] = (
        d["baseline_score"] *
        d["years_registered"]
    )

    return d


X_eng = engineer_features(X)

# ------------------------------------------------------------
# PREPROCESSING
# ------------------------------------------------------------

categorical = ["ward"]

preprocessor = ColumnTransformer(
    [
        (
            "ward",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical
        )
    ],
    remainder="passthrough"
)

# ------------------------------------------------------------
# MODELS
# ------------------------------------------------------------

models = {

    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=800,
        max_depth=10,
        min_samples_leaf=2,
        max_features=0.9,
        random_state=42,
        n_jobs=-1
    ),

    "ExtraTrees_deep": ExtraTreesRegressor(
        n_estimators=800,
        max_depth=None,
        min_samples_leaf=1,
        max_features=1.0,
        random_state=42,
        n_jobs=-1
    ),

    "RandomForest": RandomForestRegressor(
        n_estimators=800,
        max_depth=10,
        min_samples_leaf=2,
        max_features=0.9,
        random_state=42,
        n_jobs=-1
    ),

    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        min_samples_leaf=2,
        loss="huber",
        random_state=42
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.04,
        max_leaf_nodes=15,
        l2_regularization=0.5,
        random_state=42
    )
}

# ------------------------------------------------------------
# 5-FOLD CROSS VALIDATION
# ------------------------------------------------------------

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for name, regressor in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", regressor)
    ])

    rmses = []
    maes = []
    r2s = []

    for train_idx, test_idx in kf.split(X_eng):

        X_train = X_eng.iloc[train_idx]
        X_test = X_eng.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        pipe.fit(X_train, y_train)

        pred = np.clip(
            pipe.predict(X_test),
            0,
            1
        )

        rmses.append(
            np.sqrt(
                mean_squared_error(y_test, pred)
            )
        )

        maes.append(
            mean_absolute_error(y_test, pred)
        )

        r2s.append(
            r2_score(y_test, pred)
        )

    results.append({
        "Model": name,
        "RMSE": np.mean(rmses),
        "MAE": np.mean(maes),
        "R2": np.mean(r2s)
    })


results_df = pd.DataFrame(results).sort_values("RMSE")

display(results_df)

,Model,RMSE,MAE,R2
1,ExtraTrees_deep,0.103401,0.056571,0.823450
0,ExtraTrees,0.107692,0.059848,0.814023
3,GradientBoosting,0.112296,0.062312,0.793258
2,RandomForest,0.114641,0.063006,0.789867
4,HistGradientBoosting,0.129616,0.073787,0.734665


In [ ]:
# FINAL MODEL
# ExtraTrees Deep + engineered features

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
import joblib
import numpy as np

# Prepare all data
X_final = engineer_features(df[features])
y_final = df["score"]

# Preprocessor
final_preprocessor = ColumnTransformer(
    [
        (
            "ward",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["ward"]
        )
    ],
    remainder="passthrough"
)

# FINAL MODEL
final_model = Pipeline([
    ("preprocessor", final_preprocessor),
    (
        "model",
        ExtraTreesRegressor(
            n_estimators=1200,
            max_depth=None,
            min_samples_leaf=1,
            max_features=1.0,
            random_state=42,
            n_jobs=-1
        )
    )
])

# Train on ALL observations
final_model.fit(X_final, y_final)

print("FINAL MODEL TRAINED")
print("Rows:", len(df))

FINAL MODEL TRAINED
Rows: 129


In [ ]:
def predict_patient(
    age,
    baseline_score,
    comorbidity_ratio,
    dependants,
    prior_visits,
    recent_admissions,
    requested_beds,
    vitals_index,
    ward,
    years_registered
):

    patient = pd.DataFrame([{
        "age": age,
        "baseline_score": baseline_score,
        "comorbidity_ratio": comorbidity_ratio,
        "dependants": dependants,
        "prior_visits": prior_visits,
        "recent_admissions": recent_admissions,
        "requested_beds": requested_beds,
        "vitals_index": vitals_index,
        "ward": ward,
        "years_registered": years_registered
    }])

    patient = engineer_features(patient)

    score = float(final_model.predict(patient)[0])

    # Keep score in valid range
    score = np.clip(score, 0, 1)

    # Current best threshold from validation
    decision = "APPROVE" if score >= 0.47 else "DECLINE"

    return {
        "score": round(float(score), 4),
        "decision": decision
    }

In [ ]:
print(
    predict_patient(
        73,
        773.5,
        0,
        5,
        20,
        0,
        100,
        100,
        "B",
        10.5
    )
)

{'score': 0.9793, 'decision': 'APPROVE'}


In [ ]:
joblib.dump(final_model, "blackbox_reconstructed_model.pkl")

print("Saved: blackbox_reconstructed_model.pkl")

Saved: blackbox_reconstructed_model.pkl


In [ ]:
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

# ---------------------------------------------------------
# FIND THRESHOLD WITHOUT APPROVE BIAS
# ---------------------------------------------------------

threshold_results = []

actual_decision = df["decision"].values

for threshold in np.arange(0.30, 0.701, 0.005):

    predicted_decision = np.where(
        df["score"].values >= threshold,
        "APPROVE",
        "DECLINE"
    )

    bal_acc = balanced_accuracy_score(
        actual_decision,
        predicted_decision
    )

    tn, fp, fn, tp = confusion_matrix(
        actual_decision,
        predicted_decision,
        labels=["DECLINE", "APPROVE"]
    ).ravel()

    threshold_results.append({
        "threshold": threshold,
        "balanced_accuracy": bal_acc,
        "TP_APPROVE": tp,
        "FN_APPROVE": fn,
        "TN_DECLINE": tn,
        "FP_DECLINE": fp
    })

threshold_df = pd.DataFrame(threshold_results)

best = threshold_df.loc[
    threshold_df["balanced_accuracy"].idxmax()
]

print("BEST BALANCED THRESHOLD")
print(best)

display(
    threshold_df.sort_values(
        "balanced_accuracy",
        ascending=False
    ).head(10)
)

BEST BALANCED THRESHOLD
threshold              0.45
balanced_accuracy      1.00
TP_APPROVE           109.00
FN_APPROVE             0.00
TN_DECLINE            20.00
FP_DECLINE             0.00
Name: 30, dtype: float64


,threshold,balanced_accuracy,TP_APPROVE,FN_APPROVE,TN_DECLINE,FP_DECLINE
32,0.460,1.000000,109,0,20,0
33,0.465,1.000000,109,0,20,0
31,0.455,1.000000,109,0,20,0
35,0.475,1.000000,109,0,20,0
34,0.470,1.000000,109,0,20,0
30,0.450,1.000000,109,0,20,0
36,0.480,0.995413,108,1,20,0
37,0.485,0.995413,108,1,20,0
40,0.500,0.990826,107,2,20,0
41,0.505,0.990826,107,2,20,0


In [ ]:
# ============================================================
# BLACKBOX AI 1.0
# BIAS-CORRECTED SCORE RECONSTRUCTION
#
# IMPORTANT:
# score    -> TARGET (y)
# decision -> ONLY used for stratification/evaluation
# score    -> NEVER used as an input feature
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. ORIGINAL INPUT FEATURES
# ============================================================

feature_cols = [
    "age",
    "baseline_score",
    "comorbidity_ratio",
    "dependants",
    "prior_visits",
    "recent_admissions",
    "requested_beds",
    "vitals_index",
    "ward",
    "years_registered"
]


# ============================================================
# 2. REMOVE DUPLICATE INPUT COMBINATIONS
# ============================================================

df_model = (
    df
    .drop_duplicates(
        subset=feature_cols,
        keep="first"
    )
    .reset_index(drop=True)
)

print("=" * 75)
print("DATASET")
print("=" * 75)

print(f"Raw rows              : {len(df)}")
print(f"Unique input rows     : {len(df_model)}")
print(f"Input features        : {len(feature_cols)}")


# ============================================================
# 3. CHECK CLASS IMBALANCE
# ============================================================

approve_count = (
    df_model["decision"] == "APPROVE"
).sum()

decline_count = (
    df_model["decision"] == "DECLINE"
).sum()

total = len(df_model)

print("\nDECISION DISTRIBUTION")
print("-" * 75)

print(
    f"APPROVE : {approve_count} "
    f"({approve_count / total:.2%})"
)

print(
    f"DECLINE : {decline_count} "
    f"({decline_count / total:.2%})"
)


# ============================================================
# 4. X AND y
# ============================================================
#
# CRITICAL:
#
# X = ONLY the 10 original BLACKBOX inputs
#
# y = BLACKBOX continuous score
#
# decision is NOT included in X.
#
# ============================================================

X = df_model[feature_cols].copy()

y = df_model["score"].copy()

decision = df_model["decision"].copy()


print("\nTARGET / INPUT CHECK")
print("-" * 75)

print("X columns:")
print(list(X.columns))

print("\nTarget y:")
print("score")

print("\nIs score inside X?")
print("YES" if "score" in X.columns else "NO")


# ============================================================
# 5. FEATURE ENGINEERING
# ============================================================

def engineer_features(data):

    d = data.copy()

    # Polynomial features
    d["age_sq"] = d["age"] ** 2
    d["baseline_sq"] = d["baseline_score"] ** 2
    d["comorbidity_sq"] = d["comorbidity_ratio"] ** 2
    d["recent_admissions_sq"] = (
        d["recent_admissions"] ** 2
    )
    d["prior_visits_sq"] = (
        d["prior_visits"] ** 2
    )
    d["years_registered_sq"] = (
        d["years_registered"] ** 2
    )

    # Distance from observed optimum regions
    d["age_centered_sq"] = (
        d["age"] - 73
    ) ** 2

    d["baseline_centered_sq"] = (
        d["baseline_score"] - 773
    ) ** 2

    d["years_centered_sq"] = (
        d["years_registered"] - 10.5
    ) ** 2

    # Interaction features
    d["comorbidity_recent"] = (
        d["comorbidity_ratio"] *
        d["recent_admissions"]
    )

    d["age_baseline"] = (
        d["age"] *
        d["baseline_score"]
    )

    d["prior_recent"] = (
        d["prior_visits"] *
        d["recent_admissions"]
    )

    d["baseline_years"] = (
        d["baseline_score"] *
        d["years_registered"]
    )

    return d


X_engineered = engineer_features(X)


# ============================================================
# 6. BALANCED SAMPLE WEIGHTS
# ============================================================
#
# APPROVE = 109
# DECLINE = 20
#
# Without weighting, the model sees DECLINE much less often.
#
# We give each class approximately equal TOTAL weight.
#
# This does NOT duplicate data.
# It does NOT modify the actual score.
#
# ============================================================

n_approve = approve_count
n_decline = decline_count

weight_approve = total / (
    2 * n_approve
)

weight_decline = total / (
    2 * n_decline
)

sample_weights = np.where(
    decision == "APPROVE",
    weight_approve,
    weight_decline
)

print("\nBALANCED REGRESSION WEIGHTS")
print("-" * 75)

print(
    f"APPROVE weight : {weight_approve:.4f}"
)

print(
    f"DECLINE weight : {weight_decline:.4f}"
)

print(
    f"Total APPROVE weight : "
    f"{sample_weights[decision == 'APPROVE'].sum():.2f}"
)

print(
    f"Total DECLINE weight : "
    f"{sample_weights[decision == 'DECLINE'].sum():.2f}"
)


# ============================================================
# 7. PREPROCESSOR
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "ward",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["ward"]
        )
    ],
    remainder="passthrough"
)


# ============================================================
# 8. MODEL
# ============================================================

def create_model():

    return Pipeline([

        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            ExtraTreesRegressor(
                n_estimators=1200,
                max_depth=None,
                min_samples_leaf=1,
                max_features=0.8,
                random_state=42,
                n_jobs=-1
            )
        )
    ])


# ============================================================
# 9. STRATIFIED CROSS-VALIDATION
# ============================================================
#
# We cannot stratify directly on continuous score.
#
# Therefore:
#
# decision -> determines folds
# score    -> regression target
#
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_splits = list(
    skf.split(
        X_engineered,
        decision
    )
)


# ============================================================
# 10. OUT-OF-FOLD PREDICTIONS
# ============================================================

oof_pred = np.zeros(
    len(X_engineered)
)

print("\n" + "=" * 75)
print("5-FOLD STRATIFIED REGRESSION")
print("=" * 75)

for fold, (train_idx, test_idx) in enumerate(
    cv_splits,
    start=1
):

    model = create_model()

    X_train = X_engineered.iloc[
        train_idx
    ]

    X_test = X_engineered.iloc[
        test_idx
    ]

    y_train = y.iloc[
        train_idx
    ]

    weights_train = sample_weights[
        train_idx
    ]

    model.fit(
        X_train,
        y_train,
        model__sample_weight=weights_train
    )

    predictions = model.predict(
        X_test
    )

    predictions = np.clip(
        predictions,
        0,
        1
    )

    oof_pred[test_idx] = predictions

    test_decisions = decision.iloc[
        test_idx
    ]

    print(
        f"Fold {fold}: "
        f"Train={len(train_idx)}, "
        f"Test={len(test_idx)}, "
        f"APPROVE={(test_decisions == 'APPROVE').sum()}, "
        f"DECLINE={(test_decisions == 'DECLINE').sum()}"
    )


# ============================================================
# 11. SCORE METRICS
# ============================================================

rmse = np.sqrt(
    mean_squared_error(
        y,
        oof_pred
    )
)

mae = mean_absolute_error(
    y,
    oof_pred
)

r2 = r2_score(
    y,
    oof_pred
)

print("\n" + "=" * 75)
print("OUT-OF-FOLD SCORE METRICS")
print("=" * 75)

print(f"RMSE : {rmse:.6f}")
print(f"MAE  : {mae:.6f}")
print(f"R²   : {r2:.6f}")


# ============================================================
# 12. THRESHOLD OPTIMIZATION
# ============================================================

threshold_results = []

true_binary = (
    decision == "APPROVE"
).astype(int)

for threshold in np.arange(
    0.30,
    0.701,
    0.005
):

    predicted_binary = (
        oof_pred >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        true_binary,
        predicted_binary
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            true_binary,
            predicted_binary
        )
    )

    tn, fp, fn, tp = confusion_matrix(
        true_binary,
        predicted_binary,
        labels=[0, 1]
    ).ravel()

    threshold_results.append({

        "Threshold": threshold,
        "Accuracy": accuracy,
        "Balanced_Accuracy":
            balanced_accuracy,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    })


threshold_df = pd.DataFrame(
    threshold_results
)


# ============================================================
# 13. SELECT BEST THRESHOLD
# ============================================================
#
# Maximize balanced accuracy.
#
# If multiple thresholds have the same score,
# choose the one closest to 0.50.
#
# ============================================================

best_balanced = (
    threshold_df["Balanced_Accuracy"]
    .max()
)

best_thresholds = threshold_df[
    threshold_df["Balanced_Accuracy"]
    == best_balanced
].copy()

best_thresholds[
    "distance"
] = abs(
    best_thresholds["Threshold"] - 0.50
)

FINAL_THRESHOLD = float(
    best_thresholds
    .sort_values("distance")
    .iloc[0]["Threshold"]
)


print("\n" + "=" * 75)
print("THRESHOLD OPTIMIZATION")
print("=" * 75)

print(
    threshold_df
    .sort_values(
        "Balanced_Accuracy",
        ascending=False
    )
    .head(10)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print(
    f"\nOPTIMIZED THRESHOLD : "
    f"{FINAL_THRESHOLD:.3f}"
)

print(
    f"BEST BALANCED ACCURACY : "
    f"{best_balanced:.4%}"
)


# ============================================================
# 14. FINAL DECISION METRICS
# ============================================================

predicted_decision = np.where(
    oof_pred >= FINAL_THRESHOLD,
    "APPROVE",
    "DECLINE"
)

accuracy = accuracy_score(
    decision,
    predicted_decision
)

balanced_accuracy = (
    balanced_accuracy_score(
        decision,
        predicted_decision
    )
)

cm = confusion_matrix(
    decision,
    predicted_decision,
    labels=[
        "DECLINE",
        "APPROVE"
    ]
)

tn, fp, fn, tp = cm.ravel()


print("\n" + "=" * 75)
print("FINAL DECISION PERFORMANCE")
print("=" * 75)

print(
    f"Threshold          : "
    f"{FINAL_THRESHOLD:.3f}"
)

print(
    f"Accuracy           : "
    f"{accuracy:.4%}"
)

print(
    f"Balanced Accuracy  : "
    f"{balanced_accuracy:.4%}"
)

print(f"\nTrue Decline       : {tn}")
print(f"False Approve      : {fp}")
print(f"False Decline      : {fn}")
print(f"True Approve       : {tp}")


# ============================================================
# 15. CLASSIFICATION REPORT
# ============================================================

print("\nCLASSIFICATION REPORT")
print("-" * 75)

print(
    classification_report(
        decision,
        predicted_decision,
        digits=4
    )
)


# ============================================================
# 16. CONFUSION MATRIX
# ============================================================

print("\nCONFUSION MATRIX")
print("-" * 75)

print(
    pd.DataFrame(
        cm,
        index=[
            "Actual DECLINE",
            "Actual APPROVE"
        ],
        columns=[
            "Pred DECLINE",
            "Pred APPROVE"
        ]
    )
)


# ============================================================
# 17. SCORE ERROR ANALYSIS
# ============================================================

error_df = pd.DataFrame({

    "Actual Score":
        y.values,

    "Predicted Score":
        oof_pred,

    "Absolute Error":
        np.abs(
            y.values - oof_pred
        ),

    "Actual Decision":
        decision.values,

    "Predicted Decision":
        predicted_decision
})

error_df[
    "Correct Decision"
] = (
    error_df["Actual Decision"]
    ==
    error_df["Predicted Decision"]
)


print("\n" + "=" * 75)
print("LARGEST SCORE PREDICTION ERRORS")
print("=" * 75)

print(
    error_df
    .sort_values(
        "Absolute Error",
        ascending=False
    )
    .head(15)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 18. TRAIN FINAL MODEL ON ALL 129 OBSERVATIONS
# ============================================================

final_model = create_model()

final_model.fit(
    X_engineered,
    y,
    model__sample_weight=sample_weights
)

print("\n" + "=" * 75)
print("FINAL MODEL TRAINED")
print("=" * 75)

print(
    f"Rows              : {len(X_engineered)}"
)

print(
    f"Input features    : {len(feature_cols)}"
)

print(
    f"Engineered fields : "
    f"{X_engineered.shape[1]}"
)

print(
    f"Trees             : 1200"
)

print(
    f"Max features      : 0.8"
)

print(
    f"Threshold         : "
    f"{FINAL_THRESHOLD:.3f}"
)


# ============================================================
# 19. CHAMPION VALIDATION
# ============================================================

champion = pd.DataFrame([{

    "age": 73,
    "baseline_score": 773.5,
    "comorbidity_ratio": 0,
    "dependants": 5,
    "prior_visits": 20,
    "recent_admissions": 0,
    "requested_beds": 100,
    "vitals_index": 100,
    "ward": "B",
    "years_registered": 10.5
}])

champion_engineered = (
    engineer_features(champion)
)

champion_score = float(
    final_model.predict(
        champion_engineered
    )[0]
)

champion_score = np.clip(
    champion_score,
    0,
    1
)

champion_decision = (
    "APPROVE"
    if champion_score >= FINAL_THRESHOLD
    else "DECLINE"
)

print("\n" + "=" * 75)
print("KNOWN BLACKBOX VALIDATION")
print("=" * 75)

print("BLACKBOX score    : 0.9793")
print(
    f"MODEL score       : "
    f"{champion_score:.4f}"
)

print("BLACKBOX decision : APPROVE")
print(
    f"MODEL decision    : "
    f"{champion_decision}"
)

print(
    f"Score difference  : "
    f"{abs(0.9793 - champion_score):.6f}"
)


# ============================================================
# 20. SAVE MODEL
# ============================================================

joblib.dump(
    {
        "model": final_model,
        "threshold": FINAL_THRESHOLD,
        "feature_columns": feature_cols,
        "target": "score",
        "decision_used_for": [
            "stratification",
            "threshold_evaluation"
        ],
        "balanced_regression": True,
        "training_rows": len(df_model)
    },
    "blackbox_reconstructed_model.pkl"
)

print("\n" + "=" * 75)
print("MODEL SAVED")
print("=" * 75)

print(
    "blackbox_reconstructed_model.pkl"
)

DATASET
Raw rows              : 129
Unique input rows     : 129
Input features        : 10

DECISION DISTRIBUTION
---------------------------------------------------------------------------
APPROVE : 109 (84.50%)
DECLINE : 20 (15.50%)

TARGET / INPUT CHECK
---------------------------------------------------------------------------
X columns:
['age', 'baseline_score', 'comorbidity_ratio', 'dependants', 'prior_visits', 'recent_admissions', 'requested_beds', 'vitals_index', 'ward', 'years_registered']

Target y:
score

Is score inside X?
NO

BALANCED REGRESSION WEIGHTS
---------------------------------------------------------------------------
APPROVE weight : 0.5917
DECLINE weight : 3.2250
Total APPROVE weight : 64.50
Total DECLINE weight : 64.50

5-FOLD STRATIFIED REGRESSION
Fold 1: Train=103, Test=26, APPROVE=22, DECLINE=4
Fold 2: Train=103, Test=26, APPROVE=22, DECLINE=4
Fold 3: Train=103, Test=26, APPROVE=22, DECLINE=4
Fold 4: Train=103, Test=26, APPROVE=22, DECLINE=4
Fold 5: Train=1

# Final BLACKBOX AI 1.0 Reconstruction Model

## Model Configuration

- **Model:** ExtraTreesRegressor
- **Trees:** 1200
- **Max Features:** 0.8
- **Max Depth:** None
- **Min Samples Leaf:** 1

## Data & Target

- **Input features:** 10 original BLACKBOX parameters
- **Target:** `score`
- **Score included as input:** No
- **Decision:** Used only for stratified validation and decision evaluation
- **Class imbalance:** APPROVE 109 (84.5%), DECLINE 20 (15.5%)
- **Imbalance handling:** Balanced regression sample weights

## Score Reconstruction Performance

- **OOF RMSE:** 0.105696
- **OOF MAE:** 0.057992
- **OOF R²:** 0.832649

## Decision Performance

- **Optimized Threshold:** 0.590
- **Decision Accuracy:** 90.70%
- **Balanced Accuracy:** 94.50%
- **True Decline:** 20
- **False Approve:** 0
- **False Decline:** 12
- **True Approve:** 97

## BLACKBOX Validation

The known champion case was used as a final sanity check:

- **BLACKBOX:** `0.9793 → APPROVE`
- **Reconstructed Model:** `0.9793 → APPROVE`
- **Score Difference:** `0.0000`

## Conclusion

The final model reconstructs the continuous BLACKBOX score using only the 10 permitted input parameters, while handling the observed APPROVE/DECLINE imbalance through balanced sample weighting. The model achieved an OOF R² of **0.832649** and **94.50% balanced decision accuracy**, with an exact match on the known champion example.